In [ ]:
!pip install node2vec networkx pandas scikit-learn

!wget https://snap.stanford.edu/data/twitch_gamers.zip
!unzip -o twitch_gamers.zip

In [ ]:
import pandas as pd
import networkx as nx

print("1. Đang đọc dữ liệu...")
edges = pd.read_csv('large_twitch_edges.csv')
features = pd.read_csv('large_twitch_features.csv')

print("2. Đang tạo đồ thị gốc...")
G = nx.from_pandas_edgelist(edges, source='numeric_id_1', target='numeric_id_2')
print(f" -> Đồ thị gốc có: {G.number_of_nodes()} Streamer")

print("3. Đang lọc đồ thị thu gọn...")
core_nodes = [node for node, degree in dict(G.degree()).items() if degree >= 300]
G_small = G.subgraph(core_nodes).copy()
print(f" -> Đồ thị thu gọn có: {G_small.number_of_nodes()} Streamer (Dùng để chạy Node2vec)")

In [ ]:
from node2vec import Node2Vec
import numpy as np

print("Bắt đầu cho kiến đi dạo (Random Walks)...")
node2vec_model = Node2Vec(G_small, dimensions=64, walk_length=10, num_walks=3, workers=4, quiet=True)

print("Đang huấn luyện để trích xuất đặc trưng...")
model = node2vec_model.fit(window=5, min_count=1, batch_words=4)

print("Đang đóng gói vector...")
X = np.array([model.wv[str(node)] for node in G_small.nodes()])

print("Xong! Kích thước ma trận đặc trưng X là:", X.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("1. Đang đồng bộ hóa Nhãn (Ngôn ngữ)...")
features_small = features[features['numeric_id'].isin(G_small.nodes())]
labels_dict = dict(zip(features_small['numeric_id'], features_small['language']))

y_labels_text = [labels_dict[node] for node in G_small.nodes()]

le = LabelEncoder()
y = le.fit_transform(y_labels_text)

print("2. Đang huấn luyện AI phân loại...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

print("\n====== BẢNG ĐIỂM DỰ ĐOÁN NGÔN NGỮ ======\n")
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))